In [1]:
!pip install fastapi uvicorn pyngrok transformers accelerate torch pillow

In [2]:
from pyngrok import ngrok
from google.colab import userdata

ngrok_auth_token = userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(ngrok_auth_token)

In [3]:

from fastapi import FastAPI
from pydantic import BaseModel
from transformers import pipeline
import torch

pipe = pipeline(
    task="image-text-to-text",
    model="google/medgemma-4b-it",
    torch_dtype=torch.bfloat16,
    device=0,
)

app = FastAPI()

class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 512

@app.post("/generate")
def generate(body: GenerateRequest):
    output = pipe(
        text=[
            {
                "role": "user",
                "content": [{"type": "text", "text": body.prompt}],
            }
        ],
        max_new_tokens=body.max_new_tokens,
    )
    generated = output[0].get("generated_text", output[0])
    if isinstance(generated, list):
        text = generated[-1].get("content", str(generated[-1]))
    else:
        text = str(generated)
    return {"text": text}


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [4]:
import uvicorn
import threading

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8001)

thread = threading.Thread(target=run_server)
thread.start()


In [5]:
public_url = ngrok.connect(8001)
print(public_url)


INFO:     Started server process [17824]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


NgrokTunnel: "https://enlisted-flagship-unfixable.ngrok-free.dev" -> "http://localhost:8001"
